In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import os 
import json

load_dotenv(override=True)

print(os.getenv("OPENROUTER_API_KEY"))
openai = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))

"""
I'm create an ai agent that can take a tasks and create a todo for it.
"""

todo = []


# tools for create todo
def create_todo(tasks: list[str]):
  todo.extend(tasks)
  return tasks


tools = [
  {
    "type": "function",
    "function": {
      "name": "create_todo",
      "description": "Records a todo list",
      "parameters": {
        "type": "object",
        "properties": {
          "tasks": {
            "type": "array",
            "items": {
              "type": "string",
              "description": "A single task name to add to the todo list."
            },
            "description": "A list of task names to add to the todo list."
          }
        },
        "required": ["tasks"],
        "additionalProperties": False
      }
    }
  }
]

def handle_tool_calls(tool_calls):
    print(tool_calls)
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results


def agent(message: str):
  messages=[
      {"role": "system", "content": "You are a helpful assistant. When the user wants to do something or needs steps to complete a task, you MUST use the create_todo tool to record the tasks. Always call the tool — never just list tasks in text."},
      {"role": "user", "content": message}
    ]
  
  done = False 
  while not done:
    response = openai.chat.completions.create(
      model="openai/gpt-4o-mini",
      messages=messages, 
      tools=tools,
    )
    
    finish_reason = response.choices[0].finish_reason
    if finish_reason == "tool_calls":
      message = response.choices[0].message
      tool_calls = message.tool_calls
      results = handle_tool_calls(tool_calls)
      messages.append(message)
      messages.extend(results)
    else:
      done = True
      return response.choices[0].message.content




response =agent("Need a website for my company")

Markdown(response)
